# 🚀 Voyager OGM: Interactive Graph & Records Explorer (Jupyter & VS Code Notebooks)

This notebook demonstrates how to:
1. **Define OGM Node & Relationship models** using decorators (`@node`, `@relationship`).
2. **Ingest data from multiple formats**: raw Python dictionaries, CSV strings, and Polars DataFrames.
3. **Execute multiple graph queries** across domains: Org Hierarchy, IT Infrastructure Dependencies, and Fraud Detection.
4. **Render rich interactive visualizers** (`GraphViewer`, `query.show()`, `session.explore()`) with Force-Directed physics, Inspector sidebar, Records Table, and Multi-Dialect code generator.

In [1]:
import polars as pl
from voyager_ogm import (
    Field,
    GraphViewer,
    MockBridge,
    Node,
    Query,
    Relationship,
    Session,
    node,
    relationship,
)

print("Voyager OGM imported successfully!")

Voyager OGM imported successfully!


## 1. Define OGM Models & Ingest Data from Multiple Sources
We define our entity models and create datasets from:
- **Python Dictionaries** (Team Collaboration Network)
- **CSV data parsed by Polars** (Cloud Infrastructure & Microservice Topology)

In [2]:
# OGM Model Definitions
@node(label="Employee")
class Employee(Node):
    emp_id: str = Field(primary_key=True)
    name: str = Field()
    department: str = Field()
    role: str = Field()


@node(label="Project")
class Project(Node):
    project_id: str = Field(primary_key=True)
    title: str = Field()
    priority: str = Field()


@relationship(type_name="WORKS_ON")
class WorksOn(Relationship):
    allocation_pct: int = Field()


@relationship(type_name="MANAGES")
class Manages(Relationship):
    since: int = Field()


# Dataset A: Team Collaboration Records (Dictionary)
team_records = [
    {"source": "Alice (VP Eng)", "target": "Bob (Tech Lead)", "rel": "MANAGES", "since": 2021},
    {"source": "Alice (VP Eng)", "target": "Charlie (Staff Eng)", "rel": "MANAGES", "since": 2022},
    {"source": "Bob (Tech Lead)", "target": "Voyager OGM", "rel": "WORKS_ON", "allocation_pct": 80},
    {
        "source": "Charlie (Staff Eng)",
        "target": "Voyager OGM",
        "rel": "WORKS_ON",
        "allocation_pct": 50,
    },
    {
        "source": "Charlie (Staff Eng)",
        "target": "Titan Engine",
        "rel": "WORKS_ON",
        "allocation_pct": 50,
    },
    {
        "source": "Diana (Principal)",
        "target": "Titan Engine",
        "rel": "WORKS_ON",
        "allocation_pct": 100,
    },
]

# Dataset B: Microservice & Cloud Infrastructure (CSV -> Polars DataFrame)
csv_data = """source,target,rel,bandwidth_gbps,environment
WebServer_01,Auth_Gateway,CALLS,10,Production
WebServer_02,Auth_Gateway,CALLS,10,Production
Auth_Gateway,User_DB_Primary,QUERIES,40,Production
User_DB_Primary,User_DB_Replica,REPLICATES_TO,100,Production
Payment_Service,User_DB_Primary,QUERIES,40,Production
Payment_Service,Fraud_Engine,EVALUATES,25,Production
Fraud_Engine,Redis_Cache,READS,80,Production
Analytics_Worker,Redis_Cache,CONSUMES,15,Staging
"""
infra_df = pl.read_csv(csv_data.encode("utf-8"))

# Initialize graph session
mock_bridge = MockBridge()
session = Session(bridge=mock_bridge, dialect="cypher")
print("Datasets and Session initialized!")

Datasets and Session initialized!


## 2. Query 1: Exploring Team Hierarchy & Project Allocations
Visualize dictionary records directly using `GraphViewer.from_records()`.
- Switch between **Graph View** 🌐, **Records Table** 📋, and **Compiled Dialect** 📜.
- Click any node (`Alice`, `Bob`, `Voyager OGM`) to open the **Entity Inspector** drawer.

In [3]:
viewer_team = GraphViewer.from_records(
    raw_records=team_records,
    source_key="source",
    target_key="target",
    edge_label_key="rel",
    height="520px",
)
viewer_team

## 3. Query 2: Microservice & Cloud Infrastructure Topology
Visualize dependencies loaded from the Polars DataFrame with bandwidth metrics.

In [7]:
viewer_infra = GraphViewer.from_polars(
    infra_df,
    source_col="source",
    target_col="target",
    edge_label_col="rel",
    height="540px",
    theme="dark",
)
viewer_infra

## 4. Query 3: Multi-Hop Fluent OGM Query with `.show(session)`
Build a multi-hop graph query using Voyager's fluent AST builder and inspect results instantly.

In [5]:
e = Employee(alias="e")
m = Manages(alias="m")
sub = Employee(alias="sub")
w = WorksOn(alias="w")
p = Project(alias="p")

query = (
    Query.match(e)
    .to(m)
    .node(sub)
    .to(w)
    .node(p)
    .where(e.department == "Engineering")
    .return_(manager=e.name, lead=sub.name, project=p.title)
)

# Queue simulated live database results
mock_bridge.queue_result(
    [
        {"source": "Alice (VP)", "target": "Bob (Lead)", "rel": "MANAGES"},
        {"source": "Bob (Lead)", "target": "Voyager OGM", "rel": "WORKS_ON"},
        {"source": "Alice (VP)", "target": "Charlie (Staff)", "rel": "MANAGES"},
        {"source": "Charlie (Staff)", "target": "Titan Engine", "rel": "WORKS_ON"},
    ]
)

# Visualizes query results + compiled multi-dialect SQL/Cypher statements
query.show(session=session)

## 5. Query 4: Live Session Database Exploration (`session.explore()`)
Explore arbitrary graph subgraphs and fraud rings directly through the active session.

In [6]:
mock_bridge.queue_result(
    [
        {"source": "Order_#8841", "target": "Account_A", "rel": "CHARGED_TO", "amount": 1250.0},
        {"source": "Account_A", "target": "IP_192.168.1.50", "rel": "LOGGED_FROM", "risk": "High"},
        {"source": "Account_B", "target": "IP_192.168.1.50", "rel": "LOGGED_FROM", "risk": "High"},
        {
            "source": "Account_B",
            "target": "Card_Ending_9912",
            "rel": "ASSOCIATED_WITH",
            "status": "Flagged",
        },
    ]
)

session.explore("MATCH (o:Order)-[r]->(target) RETURN o, r, target LIMIT 50")